# YOLO 物件偵測訓練

In [1]:
PRETRAIN_MODEL = "yolo11n.pt"
COCO_DIR = "/home/jianhua/Desktop/dataset/SeaDroneSee_MOT_ann"
YOLO_LABEL_DIR = "/home/jianhua/Desktop/dataset/yolo_obj_detect"
DATA_YAML = "data.yaml"

# PRETRAIN_MODEL = "yolo11n.pt"
# LABEL_DIR = "/home/jianhua/Desktop/dataset/mot_annotations"
# DATASET_DIR = "/home/jianhua/Desktop/dataset/SeaDronesSee_MOT"
# YOLO_LABEL_DIR = "/home/jianhua/Desktop/dataset/yolo_obj_detect"
# DATA_YAML = "data.yaml"

In [2]:
import yaml
import os

def load_yolo_data(yaml_path):
    # 檢查檔案是否存在
    if not os.path.exists(yaml_path):
        print(f"找不到檔案: {yaml_path}")
        return None

    with open(yaml_path, 'r', encoding='utf-8') as f:
        try:
            # 使用 safe_load 避免執行惡意代碼，這是處理 YAML 的標準做法
            data = yaml.safe_load(f)
            return data
        except yaml.YAMLError as exc:
            print(f"讀取 YAML 時發生錯誤: {exc}")
            return None

config = load_yolo_data(DATA_YAML)

if config:
    print("--- 成功讀取資料集配置 ---")
    
    # 1. 取得類別數量
    NUM_CLASSES = config.get('nc', 0)
    print(f"類別數量 (nc): {NUM_CLASSES}")
    
    # 取得 DATASET 路徑
    DATASET_DIR = config.get('path', '')

    # 2. 取得類別名稱列表
    CLASS_NAMES = config.get('names', [])
    print(f"類別名稱: {CLASS_NAMES}")
    
    # 3. 取得訓練/驗證路徑
    train_path = config.get('train', '未設定')
    val_path = config.get('val', '未設定')

    TRAIN_IMG_DIR = os.path.join(DATASET_DIR, train_path)
    VAL_IMG_DIR   = os.path.join(DATASET_DIR, val_path)

    print(f"訓練集路徑: {TRAIN_IMG_DIR}")
    print(f"驗證集路徑: {VAL_IMG_DIR}")

    # 如果 names 是字典格式 {0: 'person', 1: 'dog'}，可以這樣處理
    if isinstance(CLASS_NAMES, dict):
        class_list = list(CLASS_NAMES.values())
        print(f"轉換後的類別列表: {class_list}")

--- 成功讀取資料集配置 ---
類別數量 (nc): 4
類別名稱: {0: 'swimmer', 1: 'swimmer_with_life_jacket', 2: 'boat', 3: 'life_jacket'}
訓練集路徑: /home/jianhua/Desktop/dataset/SeaDronesSee_MOT/images/train
驗證集路徑: /home/jianhua/Desktop/dataset/SeaDronesSee_MOT/images/val
轉換後的類別列表: ['swimmer', 'swimmer_with_life_jacket', 'boat', 'life_jacket']


## 轉換格式 COCO -> YOLO

In [3]:
import json
import os

def remap_coco_categories(input_json: str, output_json: str):
    with open(input_json) as f:
        data = json.load(f)

    categories = sorted(data["categories"], key=lambda x: x["id"])
    old_to_new = {cat["id"]: new_idx+1 for new_idx, cat in enumerate(categories)}

    print(old_to_new)

    # ── 先改 annotations（用舊 id 查 mapping）─────────────
    for ann in data["annotations"]:
        ann["category_id"] = old_to_new[ann["category_id"]]

    # ── 再改 categories 的 id（避免上面查到新 id）─────────
    for cat in data["categories"]:
        cat["id"] = old_to_new[cat["id"]]

    with open(output_json, "w") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"remapping 完成：{len(categories)} 個類別，新 index 0 ~ {len(categories) - 1}")
    print(f"輸出：{output_json}")

old_to_new = remap_coco_categories(
    input_json  = os.path.join(COCO_DIR ,"train.json"),
    output_json = os.path.join(COCO_DIR ,"train.json"),
)

old_to_new = remap_coco_categories(
    input_json  = os.path.join(COCO_DIR ,"val.json"),
    output_json = os.path.join(COCO_DIR ,"val.json"),
)

{1: 1, 2: 2, 3: 3, 4: 4}
remapping 完成：4 個類別，新 index 0 ~ 3
輸出：/home/jianhua/Desktop/dataset/SeaDroneSee_MOT_ann/train.json
{1: 1, 2: 2, 3: 3, 4: 4}
remapping 完成：4 個類別，新 index 0 ~ 3
輸出：/home/jianhua/Desktop/dataset/SeaDroneSee_MOT_ann/val.json


In [4]:
from ultralytics.data.converter import convert_coco

convert_coco(
    labels_dir=COCO_DIR,
    save_dir=YOLO_LABEL_DIR
)

Annotations /home/jianhua/Desktop/dataset/SeaDroneSee_MOT_ann/train.json: 100% ━━━━━━━━━━━━ 27226/27226 7.6Kit/s 3.6s0.1s
Annotations /home/jianhua/Desktop/dataset/SeaDroneSee_MOT_ann/val.json: 100% ━━━━━━━━━━━━ 8579/8579 8.4Kit/s 1.0s0.0s
COCO data converted successfully.
Results saved to /home/jianhua/Desktop/dataset/yolo_obj_detect


In [5]:
import os
import shutil

src_labels = os.path.join(YOLO_LABEL_DIR, "labels")
dst_labels = os.path.join(DATASET_DIR, "labels")

# 1️⃣ 如果 DATASET_DIR 已有 labels 就刪掉
if os.path.exists(dst_labels):
    shutil.rmtree(dst_labels)

# 2️⃣ 搬移 labels 資料夾
shutil.move(src_labels, dst_labels)

# 3️⃣ 刪掉整個 YOLO_LABEL_DIR
if os.path.exists(YOLO_LABEL_DIR):
    shutil.rmtree(YOLO_LABEL_DIR)

print("labels moved and YOLO_LABEL_DIR removed.")

# ================= 新增的部分 =================

# 定義 classes.txt 的來源與目標路徑
src_classes = "classes.txt" # 當下目錄下的 classes.txt
dst_train_dir = os.path.join(DATASET_DIR, "labels", "train")
dst_val_dir = os.path.join(DATASET_DIR, "labels", "val")

# 確保目標資料夾 train 和 val 存在 (避免 shutil.copy 找不到資料夾報錯)
os.makedirs(dst_train_dir, exist_ok=True)
os.makedirs(dst_val_dir, exist_ok=True)

# 4️⃣ 複製 classes.txt 到 train 與 val 資料夾
if os.path.exists(src_classes):
    shutil.copy(src_classes, dst_train_dir)
    shutil.copy(src_classes, dst_val_dir)
    print("classes.txt successfully copied to train and val directories.")
else:
    print(f"⚠️ 警告：在當下目錄找不到 {src_classes}，略過複製步驟。")

labels moved and YOLO_LABEL_DIR removed.
classes.txt successfully copied to train and val directories.


In [6]:
import glob
import os

def check_yolo_label_index(label_dir: str, num_classes: int):
    label_files = glob.glob(os.path.join(label_dir, "*.txt"))
    print(f"共 {len(label_files)} 個標籤檔，NUM_CLASSES = {num_classes}\n")

    errors = []
    for lbl_path in label_files:
        with open(lbl_path) as f:
            for line_no, line in enumerate(f, 1):
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                if cls_id < 0 or cls_id >= num_classes:
                    errors.append({
                        "file":    os.path.basename(lbl_path),
                        "line":    line_no,
                        "cls_id":  cls_id,
                    })

    if errors:
        print(f"發現 {len(errors)} 筆超界標注：\n")
        print(f"{'file':>40}  {'line':>6}  {'cls_id':>8}")
        print("-" * 60)
        for e in errors[:50]:
            print(f"{e['file']:>40}  {e['line']:>6}  {e['cls_id']:>8}")
        if len(errors) > 50:
            print(f"... 還有 {len(errors) - 50} 筆")
    else:
        print("所有標注都在合法範圍內，沒有問題。")

    return errors

errors = check_yolo_label_index(os.path.join(DATASET_DIR, "labels/train"), num_classes=4)

共 27227 個標籤檔，NUM_CLASSES = 4

所有標注都在合法範圍內，沒有問題。


## 開始訓練

### YOLO Library

In [7]:
# from ultralytics import YOLO
# import os

# model = YOLO(PRETRAIN_MODEL)  # load a pretrained model (recommended for training)

# val_classes = os.path.join(DATASET_DIR, "labels/val/classes.txt")
# train_classes = os.path.join(DATASET_DIR, "labels/train/classes.txt")

# if os.path.exists(train_classes) and os.path.exists(val_classes):
#     results = model.train(data="data.yaml", epochs=10, imgsz=640, batch=32)
# else:
#     raise FileNotFoundError(f"Missing required files or directories: {val_classes} {train_classes}")

### Costum YOLO Architecture

In [8]:
# from arch.yolo11 import yolo_v11_n
# from torchinfo import summary

# # yolo_model = yolo_v11_n(num_classes=nc)
# yolo_model = yolo_v11_n(num_classes=4)

# # input_size 格式為 (Batch_size, Channels, Height, Width)
# # YOLO 通常使用 640x640 的輸入
# summary(yolo_model, input_size=(1, 3, 640, 640))

In [9]:
# """
# YOLOv11 Custom Training Script — NVIDIA DALI GPU Pipeline
# Based on: https://github.com/jahongir7174/YOLOv11-pt

# Pipeline 設計：
#   ┌─────────────────────────────────────────────────────┐
#   │  DALI GPU Pipeline (per-sample ops)                 │
#   │  file read → GPU JPEG decode → letterbox resize     │
#   │  → HSV jitter → flip  →  已在 GPU 的 tensor        │
#   └─────────────────────────────────────────────────────┘
#   ┌─────────────────────────────────────────────────────┐
#   │  Mosaic（跨 sample，DALI 不支援原生）               │
#   │  用 ExternalSource + CPU worker 組裝，              │
#   │  組好後一次性送到 GPU，避免每張圖個別傳輸           │
#   └─────────────────────────────────────────────────────┘

# 安裝：
#   pip install nvidia-dali-cuda120   # 對應 CUDA 12.x
#   pip install nvidia-dali-cuda110   # 對應 CUDA 11.x
# """

# import copy
# import csv
# import glob
# import math
# import os
# import queue
# import random
# import threading
# from types import SimpleNamespace

# import cv2
# import numpy as np
# import torch
# import tqdm
# from torch.utils import data as torch_data

# import nvidia.dali.fn as fn
# import nvidia.dali.types as types
# from nvidia.dali.pipeline import pipeline_def
# from nvidia.dali.plugin.pytorch import DALIGenericIterator, LastBatchPolicy  # ← 正確 import

# from arch.yolo11 import yolo_v11_n
# from utils import util

# # ─────────────────────────────────────────────────────────────
# # 設定
# # ─────────────────────────────────────────────────────────────

# INPUT_SIZE  = 640
# BATCH_SIZE  = 32
# EPOCHS      = 10
# WORKERS     = 8       # DALI CPU decode thread 數
# DEVICE_ID   = 0

# PARAMS = {
#     "names":         [f"{list(CLASS_NAMES.values())[i]}" for i in range(NUM_CLASSES)],
#     "min_lr":        1e-5,    # warmup 起點接近 0
#     "max_lr":        1e-1,    # peak lr
#     "momentum":      0.937,
#     "weight_decay":  1e-4,    # 降低，避免 underfit
#     "warmup_epochs": 1,       # 縮短 warmup
#     "lrf":           0.01,
#     "box":           7.5,
#     "cls":           0.5,
#     "dfl":           1.5,

#     # augment
#     "mosaic":        1.0,
#     "mix_up":        0.1,
#     "hsv_h":         0.015,
#     "hsv_s":         0.7,
#     "hsv_v":         0.4,
#     "degrees":       0.0,
#     "scale":         0.5,
#     "shear":         0.0,
#     "translate":     0.1,
#     "flip_ud":       0.0,
#     "flip_lr":       0.5,
# }

# WEIGHTS_DIR = "weights"


# # ─────────────────────────────────────────────────────────────
# # 1. 標籤讀取
# # ─────────────────────────────────────────────────────────────
 
# def load_labels(img_paths: list) -> dict:
#     result = {}
#     for img_path in img_paths:
#         lbl_path = img_path.replace(f"{os.sep}images{os.sep}",
#                                     f"{os.sep}labels{os.sep}"
#                                     ).rsplit(".", 1)[0] + ".txt"
#         if os.path.exists(lbl_path):
#             with open(lbl_path) as f:
#                 rows = [line.split() for line in f.read().strip().splitlines() if line]
#             arr = np.array(rows, dtype=np.float32) if rows else np.zeros((0, 5), np.float32)
#         else:
#             arr = np.zeros((0, 5), np.float32)
#         result[img_path] = arr
#     return result
 
 
# def scan_images(folder: str) -> list:
#     paths = []
#     for ext in ("*.jpg", "*.jpeg", "*.png"):
#         paths.extend(glob.glob(os.path.join(folder, ext)))
#     paths.sort()
#     return paths
 
 
# # ─────────────────────────────────────────────────────────────
# # 2. DALI Pipeline
# # ─────────────────────────────────────────────────────────────
 
# @pipeline_def(
#     batch_size=BATCH_SIZE,
#     num_threads=WORKERS,
#     device_id=DEVICE_ID,
#     prefetch_queue_depth=2,
# )
# def yolo_dali_pipeline(img_paths: list, augment: bool = True):
#     encoded, indices = fn.readers.file(
#         files          = img_paths,
#         random_shuffle = augment,
#         name           = "reader",
#     )
 
#     images = fn.decoders.image(
#         encoded,
#         device      = "mixed",
#         output_type = types.BGR,
#     )
 
#     images = fn.resize(
#         images,
#         device        = "gpu",
#         resize_longer = INPUT_SIZE,
#         interp_type   = types.INTERP_LINEAR,
#     )
 
#     images = fn.pad(
#         images,
#         device     = "gpu",
#         fill_value = 114,
#         shape      = [INPUT_SIZE, INPUT_SIZE],
#         axes       = [0, 1],
#     )
 
#     if augment:
#         hue_delta = fn.random.uniform(
#             range=(-PARAMS["hsv_h"] * 180, PARAMS["hsv_h"] * 180))
#         sat_scale = fn.random.uniform(
#             range=(1 - PARAMS["hsv_s"], 1 + PARAMS["hsv_s"]))
#         val_scale = fn.random.uniform(
#             range=(1 - PARAMS["hsv_v"], 1 + PARAMS["hsv_v"]))
 
#         images = fn.hsv(
#             images,
#             device     = "gpu",
#             hue        = hue_delta,
#             saturation = sat_scale,
#             value      = val_scale,
#         )
 
#         do_flip = fn.random.coin_flip(probability=PARAMS["flip_lr"])
#         images  = fn.flip(images, device="gpu", horizontal=do_flip)
 
#     images = fn.transpose(images, device="gpu", perm=[2, 0, 1])
 
#     return images, indices.gpu()
 
 
# # ─────────────────────────────────────────────────────────────
# # 3. Mosaic helper（CPU）
# # ─────────────────────────────────────────────────────────────

# def build_mosaic(img_paths, labels_map, size=INPUT_SIZE):
#     s      = size
#     # center 必須讓四個象限都有非零大小：限制在 [s//4, s*3//4]
#     yc     = int(random.uniform(s // 4, s * 3 // 4))
#     xc     = int(random.uniform(s // 4, s * 3 // 4))
#     canvas = np.full((s * 2, s * 2, 3), 114, dtype=np.uint8)
#     all_lbs = []
 
#     placements = [
#         (0,  0,  xc,    yc   ),   # top-left
#         (0,  xc, s * 2, yc   ),   # top-right
#         (yc, 0,  xc,    s * 2),   # bottom-left
#         (yc, xc, s * 2, s * 2),   # bottom-right
#     ]
 
#     for (r1, c1, c2, r2) in placements:
#         ph, pw = r2 - r1, c2 - c1
#         if ph <= 0 or pw <= 0:
#             continue
 
#         idx      = random.randrange(len(img_paths))
#         img_path = img_paths[idx]
#         img      = cv2.imread(img_path)
#         if img is None:
#             continue
 
#         img_r = cv2.resize(img, (pw, ph), interpolation=cv2.INTER_LINEAR)
#         canvas[r1:r2, c1:c2] = img_r
 
#         lbs = labels_map[img_path].copy()
#         if len(lbs):
#             lbs[:, 1] = lbs[:, 1] * pw + c1
#             lbs[:, 2] = lbs[:, 2] * ph + r1
#             lbs[:, 3] = lbs[:, 3] * pw
#             lbs[:, 4] = lbs[:, 4] * ph
#             all_lbs.append(lbs)
 
#     # 裁切回 (s, s)，用 clip 確保不超出 canvas 邊界
#     y1 = max(0, yc - s // 2)
#     y2 = min(s * 2, yc + s // 2)
#     x1 = max(0, xc - s // 2)
#     x2 = min(s * 2, xc + s // 2)
 
#     crop = canvas[y1:y2, x1:x2]
#     if crop.shape[0] == 0 or crop.shape[1] == 0:
#         # 極端情況 fallback：直接回傳空標籤
#         return canvas[:s, :s].copy(), np.zeros((0, 5), np.float32)
 
#     mosaic_img = cv2.resize(crop, (s, s), interpolation=cv2.INTER_LINEAR)
 
#     if all_lbs:
#         lbs = np.concatenate(all_lbs, 0)
#         # 對齊裁切偏移
#         lbs[:, 1] -= x1
#         lbs[:, 2] -= y1
#         # 縮放比例（crop → s）
#         scale_x = s / max(x2 - x1, 1)
#         scale_y = s / max(y2 - y1, 1)
#         lbs[:, 1] *= scale_x
#         lbs[:, 2] *= scale_y
#         lbs[:, 3] *= scale_x
#         lbs[:, 4] *= scale_y
#         # 正規化
#         lbs[:, 1:] /= s
#         lbs[:, 1:] = np.clip(lbs[:, 1:], 0.001, 0.999)
#     else:
#         lbs = np.zeros((0, 5), np.float32)
 
#     return mosaic_img, lbs
 
 
# # ─────────────────────────────────────────────────────────────
# # 4. YOLODALILoader
# # ─────────────────────────────────────────────────────────────
 
# class YOLODALILoader:
#     def __init__(self, img_folder: str, augment: bool = True, mosaic: bool = False):
#         self.augment    = augment
#         self.img_paths  = scan_images(img_folder)
#         self.labels_map = load_labels(self.img_paths)
#         self.n          = len(self.img_paths)
#         self.mosaic     = mosaic
 
#         self._pipe = yolo_dali_pipeline(
#             img_paths = self.img_paths,
#             augment   = augment,
#         )
#         self._pipe.build()
 
#         # ── 修正：使用 LastBatchPolicy enum，不用字串 ──────
#         policy = LastBatchPolicy.DROP if augment else LastBatchPolicy.PARTIAL
 
#         self._iter = DALIGenericIterator(
#             pipelines         = [self._pipe],
#             output_map        = ["images", "indices"],
#             last_batch_policy = policy,     # ← enum 值
#             auto_reset        = True,
#             reader_name       = "reader",   # 讓 DALI 自動推算 size
#         )
 
#         self._steps = math.ceil(self.n / BATCH_SIZE)
 
#     def __len__(self):
#         return self._steps
 
#     def __iter__(self):
#         for batch in self._iter:
#             images  = batch[0]["images"].float() / 255.0
#             indices = batch[0]["indices"].cpu().numpy().flatten().astype(int)
 
#             all_cls, all_box, all_idx = [], [], []
 
#             for i, img_idx in enumerate(indices):
#                 img_path = self.img_paths[img_idx % self.n]
 
#                 if self.augment and self.mosaic and random.random() < PARAMS["mosaic"]:
#                     _, lbs = build_mosaic(self.img_paths, self.labels_map)
#                 else:
#                     lbs = self.labels_map[img_path].copy()
 
#                 if len(lbs):
#                     all_cls.append(torch.from_numpy(lbs[:, 0:1]).float())
#                     all_box.append(torch.from_numpy(lbs[:, 1:5]).float())
#                     all_idx.append(torch.full((len(lbs),), i, dtype=torch.long))
 
#             if all_cls:
#                 targets = {
#                     "idx": torch.cat(all_idx, 0).cuda(),
#                     "cls": torch.cat(all_cls, 0).cuda(),
#                     "box": torch.cat(all_box, 0).cuda(),
#                 }
#             else:
#                 targets = {
#                     "idx": torch.zeros(0, dtype=torch.long).cuda(),
#                     "cls": torch.zeros((0, 1)).cuda(),
#                     "box": torch.zeros((0, 4)).cuda(),
#                 }
 
#             yield images, targets
 
 
# # ─────────────────────────────────────────────────────────────
# # 5. 訓練
# # ─────────────────────────────────────────────────────────────
 
# def train():
#     os.makedirs(WEIGHTS_DIR, exist_ok=True)
 
#     model = yolo_v11_n(NUM_CLASSES)
#     model.cuda()
 
#     accumulate = 1
#     params = PARAMS.copy()
#     params["weight_decay"] *= BATCH_SIZE * accumulate / 64
 
#     # optimizer = torch.optim.SGD(
#     #     util.set_params(model, params["weight_decay"]),
#     #     params["min_lr"],
#     #     params["momentum"],
#     #     nesterov=True,
#     # )

#     optimizer = torch.optim.AdamW(
#         util.set_params(model, params["weight_decay"]),
#         lr           = params["max_lr"],   # Adam 直接從 max_lr 開始，不用 min_lr 當初始值
#         betas        = (0.9, 0.999),
#         eps          = 1e-8,
#     )
 
#     ema = util.EMA(model)
 
#     if len(os.listdir(TRAIN_IMG_DIR)) == 0:
#         raise FileNotFoundError(f"找不到圖片：{TRAIN_IMG_DIR}")
#     else:
#         print(f"[Train] 找到 {len(os.listdir(TRAIN_IMG_DIR))} 張圖片")

#     if len(os.listdir(VAL_IMG_DIR)) == 0:
#         raise FileNotFoundError(f"找不到圖片：{VAL_IMG_DIR}")
#     else:
#         print(f"[VAL] 找到 {len(os.listdir(VAL_IMG_DIR))} 張圖片")

#     train_loader = YOLODALILoader(TRAIN_IMG_DIR, augment=True)
#     num_steps    = len(train_loader)
 
#     args = SimpleNamespace(epochs=EPOCHS, local_rank=0, world_size=1)
#     # scheduler = util.LinearLR(args, params, num_steps)
#     scheduler = util.CosineLR(args, params, num_steps)
#     criterion = util.ComputeLoss(model, params)
 
#     best = 0.0
 
#     with open(f"{WEIGHTS_DIR}/step.csv", "w") as log:
#         logger = csv.DictWriter(log, fieldnames=["epoch", "box", "cls", "dfl",
#                                                   "Recall", "Precision", "mAP@50", "mAP"])
#         logger.writeheader()
 
#         for epoch in range(EPOCHS):
#             model.train()
 
#             if EPOCHS - epoch == 10:
#                 train_loader.mosaic = False
 
#             avg_box = util.AverageMeter()
#             avg_cls = util.AverageMeter()
#             avg_dfl = util.AverageMeter()
 
#             print(("\n" + "%10s" * 5) % ("epoch", "memory", "box", "cls", "dfl"))
#             p_bar = tqdm.tqdm(enumerate(train_loader), total=num_steps)
 
#             for i, (samples, targets) in p_bar:
#                 step = i + num_steps * epoch
#                 scheduler.step(step, optimizer)
 
#                 optimizer.zero_grad()
#                 outputs = model(samples)
#                 loss_box, loss_cls, loss_dfl = criterion(outputs, targets)
 
#                 avg_box.update(loss_box.item(), samples.size(0))
#                 avg_cls.update(loss_cls.item(), samples.size(0))
#                 avg_dfl.update(loss_dfl.item(), samples.size(0))
 
#                 (loss_box + loss_cls + loss_dfl).backward()
#                 optimizer.step()
#                 ema.update(model)
 
#                 torch.cuda.synchronize()
 
#                 mem = f"{torch.cuda.memory_reserved() / 1e9:.3g}G"
#                 s   = ("%10s" * 2 + "%10.4g" * 3) % (
#                     f"{epoch + 1}/{EPOCHS}", mem,
#                     avg_box.avg, avg_cls.avg, avg_dfl.avg,
#                 )
#                 p_bar.set_description(s)
 
#             last = test(params, model=ema.ema)
 
#             logger.writerow({
#                 "epoch":     str(epoch + 1).zfill(3),
#                 "box":       f"{avg_box.avg:.4f}",
#                 "cls":       f"{avg_cls.avg:.4f}",
#                 "dfl":       f"{avg_dfl.avg:.4f}",
#                 "mAP":       f"{last[0]:.3f}",
#                 "mAP@50":    f"{last[1]:.3f}",
#                 "Recall":    f"{last[2]:.3f}",
#                 "Precision": f"{last[3]:.3f}",
#             })
#             log.flush()
 
#             save = {"epoch": epoch + 1, "model": copy.deepcopy(ema.ema)}
#             torch.save(save, f"{WEIGHTS_DIR}/last.pt")
#             if last[0] > best:
#                 best = last[0]
#                 torch.save(save, f"{WEIGHTS_DIR}/best.pt")
#             del save
 
#     util.strip_optimizer(f"{WEIGHTS_DIR}/best.pt")
#     util.strip_optimizer(f"{WEIGHTS_DIR}/last.pt")
 
 
# # ─────────────────────────────────────────────────────────────
# # 6. Validation
# # ─────────────────────────────────────────────────────────────
 
# @torch.no_grad()
# def test(params, model=None):
#     val_loader = YOLODALILoader(VAL_IMG_DIR, augment=False)
 
#     plot = False
#     if model is None:
#         plot  = True
#         ckpt  = torch.load(f"{WEIGHTS_DIR}/best.pt", map_location="cuda")
#         model = ckpt["model"].float().fuse()
 
#     model.half()
#     model.eval()
 
#     iou_v   = torch.linspace(0.5, 0.95, 10).cuda()
#     n_iou   = iou_v.numel()
#     m_pre   = m_rec = map50 = mean_ap = 0
#     metrics = []
 
#     p_bar = tqdm.tqdm(val_loader,
#                       desc=("%10s" * 5) % ("", "precision", "recall", "mAP50", "mAP"))
 
#     for samples, targets in p_bar:
#         samples = samples.half()
#         _, _, h, w = samples.shape
#         scale = torch.tensor((w, h, w, h)).cuda()
 
#         outputs = model(samples)
#         outputs = util.non_max_suppression(outputs)
 
#         for i, output in enumerate(outputs):
#             idx = targets["idx"] == i
#             cls = targets["cls"][idx]
#             box = targets["box"][idx]
#             metric = torch.zeros(output.shape[0], n_iou, dtype=torch.bool).cuda()
 
#             if output.shape[0] == 0:
#                 if cls.shape[0]:
#                     metrics.append((metric, *torch.zeros((2, 0)).cuda(), cls.squeeze(-1)))
#                 continue
 
#             if cls.shape[0]:
#                 target = torch.cat((cls, util.wh2xy(box) * scale), dim=1)
#                 metric = util.compute_metric(output[:, :6], target, iou_v)
 
#             metrics.append((metric, output[:, 4], output[:, 5], cls.squeeze(-1)))
 
#     metrics = [torch.cat(x, 0).cpu().numpy() for x in zip(*metrics)]
#     if len(metrics) and metrics[0].any():
#         tp, fp, m_pre, m_rec, map50, mean_ap = util.compute_ap(
#             *metrics, plot=plot, names=params["names"]
#         )
 
#     print(("%10s" + "%10.3g" * 4) % ("", m_pre, m_rec, map50, mean_ap))
#     model.float()
#     return mean_ap, map50, m_rec, m_pre
 
 
# # ─────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     util.setup_seed()
#     util.setup_multi_processes()
#     train()

In [ ]:
"""
YOLOv11 Custom Training Script
Based on: https://github.com/jahongir7174/YOLOv11-pt

新增對齊 Ultralytics 的優化：
  1. copy_paste 增強（dataset.py 已整合）
  2. 多尺度訓練（每個 batch 隨機縮放輸入大小）
  3. warmup 同時 warm up momentum
  4. top_k 從 10 → 13
"""

import copy
import csv
import os
from types import SimpleNamespace

import torch
import torch.nn.functional as F
import tqdm

from arch.yolo11 import yolo_v11_n
from utils import util
from utils.dataset import create_dataloader, Dataset

# ─────────────────────────────────────────────────────────────
# 設定
# ─────────────────────────────────────────────────────────────

INPUT_SIZE  = 640
BATCH_SIZE  = 16
EPOCHS      = 10
WORKERS     = 4

PARAMS = {
    # ── 類別 ──────────────────────────────────────────────
    "names": [f"class{i}" for i in range(NUM_CLASSES)],

    # ── 優化器 ────────────────────────────────────────────
    "min_lr":           1e-5,    # warmup 起點
    "max_lr":           1e-2,    # peak lr
    "momentum":         0.937,
    "warmup_momentum":  0.8,     # [NEW] warmup 期間 momentum 從低到高
    "weight_decay":     1e-4,
    "warmup_epochs":    1,
    "lrf":              0.01,

    # ── Loss 權重 ─────────────────────────────────────────
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,

    # ── Dataset 增強 ───────────────────────────────────────
    "mosaic":     1.0,
    "mix_up":     0.1,
    "copy_paste": 0.1,   # [NEW] copy-paste 機率
    "hsv_h":      0.015,
    "hsv_s":      0.7,
    "hsv_v":      0.4,
    "degrees":    0.0,
    "scale":      0.5,
    "shear":      0.0,
    "translate":  0.1,
    "flip_ud":    0.0,
    "flip_lr":    0.5,

    # ── 多尺度訓練 ────────────────────────────────────────
    "multi_scale": True,   # [NEW] 每個 batch 隨機縮放輸入大小
}

WEIGHTS_DIR = "weights"


# ─────────────────────────────────────────────────────────────
# Warmup Scheduler（同時 warm up lr 和 momentum）
# ─────────────────────────────────────────────────────────────

class WarmupScheduler:
    """
    Warmup 階段：lr 從 min_lr → max_lr，momentum 從 warmup_momentum → momentum
    Warmup 結束後：接原本的 LinearLR / CosineLR
    """

    def __init__(self, args, params, num_steps):
        self.warmup_steps    = int(max(params['warmup_epochs'] * num_steps, 100))
        self.min_lr          = params['min_lr']
        self.max_lr          = params['max_lr']
        self.momentum        = params['momentum']
        self.warmup_momentum = params.get('warmup_momentum', 0.8)

        # warmup 結束後接 LinearLR
        self._base = util.LinearLR(args, params, num_steps)

    def step(self, step, optimizer):
        if step < self.warmup_steps:
            # warmup：線性增加 lr，線性增加 momentum
            ratio = step / max(self.warmup_steps, 1)
            lr    = self.min_lr + (self.max_lr - self.min_lr) * ratio
            mom   = self.warmup_momentum + (self.momentum - self.warmup_momentum) * ratio
            for pg in optimizer.param_groups:
                pg['lr'] = lr
                if 'momentum' in pg:
                    pg['momentum'] = mom
        else:
            # warmup 結束後交給原本的 scheduler
            self._base.step(step, optimizer)


# ─────────────────────────────────────────────────────────────
# 訓練
# ─────────────────────────────────────────────────────────────

def train():
    os.makedirs(WEIGHTS_DIR, exist_ok=True)

    # ── DATASET ────────────────────────────────────────────
    print(f"[Train] {len(os.listdir(TRAIN_IMG_DIR))} Images")
    print(f"[VAL] {len(os.listdir(VAL_IMG_DIR))} Images")

    if len(os.listdir(TRAIN_IMG_DIR))==0 or len(os.listdir(VAL_IMG_DIR))==0:
        print("Error : Can't Found Dataset")
        return None

    # ── Model ──────────────────────────────────────────────
    model = yolo_v11_n(NUM_CLASSES)
    model.cuda()

    # ── top_k 13（對齊 Ultralytics）────────────────────────
    # ComputeLoss 內部的 Assigner 預設 top_k=10，改成 13
    # 需要在 ComputeLoss 建立前 patch
    _orig_assigner_init = util.Assigner.__init__
    def _patched_init(self, nc=80, top_k=13, alpha=0.5, beta=6.0, eps=1e-9):
        _orig_assigner_init(self, nc=nc, top_k=top_k,
                            alpha=alpha, beta=beta, eps=eps)
    util.Assigner.__init__ = _patched_init

    # ── Optimizer ──────────────────────────────────────────
    accumulate = 1
    params = PARAMS.copy()
    params["weight_decay"] *= BATCH_SIZE * accumulate / 64

    optimizer = torch.optim.AdamW(
        util.set_params(model, params["weight_decay"]),
        lr    = params["max_lr"],
        betas = (0.9, 0.999),
        eps   = 1e-8,
    )

    # ── EMA ────────────────────────────────────────────────
    ema = util.EMA(model)

    # ── DataLoader ─────────────────────────────────────────
    train_loader = create_dataloader(
        img_folder = TRAIN_IMG_DIR,
        input_size = INPUT_SIZE,
        batch_size = BATCH_SIZE,
        augment    = True,
        shuffle    = True,
        hyp_params = params,
    )
    num_steps = len(train_loader)

    # ── Scheduler ──────────────────────────────────────────
    args = SimpleNamespace(epochs=EPOCHS, local_rank=0, world_size=1)
    scheduler = WarmupScheduler(args, params, num_steps)

    # ── Loss ───────────────────────────────────────────────
    criterion = util.ComputeLoss(model, params)

    # ── Training Loop ──────────────────────────────────────
    best = 0.0
    multi_scale = params.get("multi_scale", False)

    with open(f"{WEIGHTS_DIR}/step.csv", "w") as log:
        logger = csv.DictWriter(log, fieldnames=["epoch", "box", "cls", "dfl",
                                                  "Recall", "Precision", "mAP@50", "mAP"])
        logger.writeheader()

        for epoch in range(EPOCHS):
            model.train()

            # 最後 30 epoch 關閉 mosaic 和 copy_paste
            if EPOCHS - epoch == 30:
                train_loader.dataset.mosaic = False
                params["copy_paste"] = 0.0

            avg_box = util.AverageMeter()
            avg_cls = util.AverageMeter()
            avg_dfl = util.AverageMeter()

            print(("\n" + "%10s" * 5) % ("epoch", "memory", "box", "cls", "dfl"))
            p_bar = tqdm.tqdm(enumerate(train_loader), total=num_steps)

            for i, (samples, targets) in p_bar:
                step = i + num_steps * epoch
                scheduler.step(step, optimizer)

                samples = samples.cuda().float() / 255.0

                # ── [NEW] 多尺度訓練 ───────────────────────
                if multi_scale and epoch < EPOCHS - 30:
                    # 隨機選 size，範圍 INPUT_SIZE ± 50%，步長 32
                    sz = random.randrange(
                        int(INPUT_SIZE * 0.5),
                        int(INPUT_SIZE * 1.5) + 32,
                    ) // 32 * 32
                    if sz != INPUT_SIZE:
                        samples = F.interpolate(
                            samples, size=sz,
                            mode='bilinear', align_corners=False,
                        )

                optimizer.zero_grad()
                outputs = model(samples)
                loss_box, loss_cls, loss_dfl = criterion(outputs, targets)

                avg_box.update(loss_box.item(), samples.size(0))
                avg_cls.update(loss_cls.item(), samples.size(0))
                avg_dfl.update(loss_dfl.item(), samples.size(0))

                (loss_box + loss_cls + loss_dfl).backward()
                optimizer.step()
                ema.update(model)

                torch.cuda.synchronize()

                mem = f"{torch.cuda.memory_reserved() / 1e9:.3g}G"
                s   = ("%10s" * 2 + "%10.4g" * 3) % (
                    f"{epoch + 1}/{EPOCHS}", mem,
                    avg_box.avg, avg_cls.avg, avg_dfl.avg,
                )
                p_bar.set_description(s)

            # ── Validation ─────────────────────────────────
            last = test(params, model=ema.ema)

            logger.writerow({
                "epoch":     str(epoch + 1).zfill(3),
                "box":       f"{avg_box.avg:.4f}",
                "cls":       f"{avg_cls.avg:.4f}",
                "dfl":       f"{avg_dfl.avg:.4f}",
                "mAP":       f"{last[0]:.3f}",
                "mAP@50":    f"{last[1]:.3f}",
                "Recall":    f"{last[2]:.3f}",
                "Precision": f"{last[3]:.3f}",
            })
            log.flush()

            save = {"epoch": epoch + 1, "model": copy.deepcopy(ema.ema)}
            torch.save(save, f"{WEIGHTS_DIR}/last.pt")
            if last[0] > best:
                best = last[0]
                torch.save(save, f"{WEIGHTS_DIR}/best.pt")
            del save

    util.strip_optimizer(f"{WEIGHTS_DIR}/best.pt")
    util.strip_optimizer(f"{WEIGHTS_DIR}/last.pt")


# ─────────────────────────────────────────────────────────────
# Validation
# ─────────────────────────────────────────────────────────────

@torch.no_grad()
def test(params, model=None):
    val_loader = create_dataloader(
        img_folder = VAL_IMG_DIR,
        input_size = INPUT_SIZE,
        batch_size = 4,
        augment    = False,
        shuffle    = False,
        hyp_params = params,
    )

    plot = False
    if model is None:
        plot  = True
        ckpt  = torch.load(f"{WEIGHTS_DIR}/best.pt", map_location="cuda")
        model = ckpt["model"].float().fuse()

    model.half()
    model.eval()

    iou_v   = torch.linspace(0.5, 0.95, 10).cuda()
    n_iou   = iou_v.numel()
    m_pre   = m_rec = map50 = mean_ap = 0
    metrics = []

    p_bar = tqdm.tqdm(val_loader,
                      desc=("%10s" * 5) % ("", "precision", "recall", "mAP50", "mAP"))

    for samples, targets in p_bar:
        samples = samples.cuda().half() / 255.0
        _, _, h, w = samples.shape
        scale = torch.tensor((w, h, w, h)).cuda()

        outputs = model(samples)
        outputs = util.non_max_suppression(outputs)

        for i, output in enumerate(outputs):
            idx = targets["idx"] == i
            cls = targets["cls"][idx].cuda()
            box = targets["box"][idx].cuda()
            metric = torch.zeros(output.shape[0], n_iou, dtype=torch.bool).cuda()

            if output.shape[0] == 0:
                if cls.shape[0]:
                    metrics.append((metric, *torch.zeros((2, 0)).cuda(), cls.squeeze(-1)))
                continue

            if cls.shape[0]:
                target = torch.cat((cls, util.wh2xy(box) * scale), dim=1)
                metric = util.compute_metric(output[:, :6], target, iou_v)

            metrics.append((metric, output[:, 4], output[:, 5], cls.squeeze(-1)))

    metrics = [torch.cat(x, 0).cpu().numpy() for x in zip(*metrics)]
    if len(metrics) and metrics[0].any():
        tp, fp, m_pre, m_rec, map50, mean_ap = util.compute_ap(
            *metrics, plot=plot, names=params["names"]
        )

    print(("%10s" + "%10.3g" * 4) % ("", m_pre, m_rec, map50, mean_ap))
    model.float()
    return mean_ap, map50, m_rec, m_pre


# ─────────────────────────────────────────────────────────────

import random   # noqa（放這裡避免影響上方 import 順序）

if __name__ == "__main__":
    util.setup_seed()
    util.setup_multi_processes()
    train()

成功找到 27259 張圖片。

[提示] 已自動刪除舊的快取檔案: /home/jianhua/Desktop/dataset/SeaDronesSee_MOT/images/train.cache

     epoch    memory       box       cls       dfl


/home/jianhua/Desktop/vitis-ai-pytorch/train/utils/dataset.py:421: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.1),
/home/jianhua/Desktop/vitis-ai-pytorch/train/utils/dataset.py:425: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes, min_height, min_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32,
  0%|          | 0/1704 [00:00<?, ?it/s]/home/jianhua/miniconda3/envs/torch2/lib/python3.9/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /opt/conda/conda-bld/pytorch_1678402374358/work/aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
      1/10     3.25G     1.897     1.489     1.416: 100%|██████████| 1704/1704 [06:54<00:00,  4.12it/s]


成功找到 8587 張圖片。


           precision    recall     mAP50       mAP: 100%|██████████| 2147/2147 [01:48<00:00, 19.85it/s]


               0.661     0.515     0.542     0.261

     epoch    memory       box       cls       dfl


      2/10     3.26G     1.435    0.8461     1.103:   4%|▍         | 67/1704 [00:14<04:28,  6.09it/s]